# Task 2: U-Net Coronary Vessel Segmentation (Arcade Dataset)

This notebook trains a U-Net model with a ResNet34 backbone using `segmentation-models-pytorch` to segment blood vessels in coronary angiogram images.

In [ ]:
# Mount Google Drive to access your dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install segmentation-models-pytorch albumentations opencv-python

In [ ]:
import os
import cv2
import numpy as np
from glob import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

In [ ]:
# ----- DIRECTORY CONFIGURATION -----
# Update these paths to match your dataset structure in Google Drive
TRAIN_IMG_DIR = "/content/drive/MyDrive/arcade/train/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/arcade/train/masks"
VAL_IMG_DIR = "/content/drive/MyDrive/arcade/val/images"
VAL_MASK_DIR = "/content/drive/MyDrive/arcade/val/masks"

IMAGE_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 40
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transforms=None):
        self.image_paths = sorted(glob(os.path.join(image_dir, "*")))
        self.mask_paths = sorted(glob(os.path.join(mask_dir, "*")))
        assert len(self.image_paths) == len(self.mask_paths), f"Images ({len(self.image_paths)})/masks ({len(self.mask_paths)}) mismatch"
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx], cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

        # Ensure mask is binary (0/1)
        mask = (mask > 127).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]
        else:
            img = ToTensorV2()(image=img)["image"]
            mask = torch.tensor(mask).unsqueeze(0).float()

        if isinstance(mask, torch.Tensor):
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
        else:
            mask = torch.tensor(mask).unsqueeze(0).float()

        return img.float(), mask.float()

In [ ]:
# Define Augmentations
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=15, p=0.4),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0)),
    ToTensorV2()
])

train_ds = SegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transforms=train_transform)
val_ds   = SegmentationDataset(VAL_IMG_DIR, VAL_MASK_DIR, transforms=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Initialize Model, Optimizer, Loss, and Scheduler
model = smp.Unet(
    encoder_name="resnet34", 
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=1, 
    activation=None
).to(DEVICE)

bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, eps=1e-7):
    pred = torch.sigmoid(pred)
    num = 2 * (pred * target).sum(dim=(2,3))
    den = (pred + target).sum(dim=(2,3))
    loss = 1 - (num + eps) / (den + eps)
    return loss.mean()

def combined_loss(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

In [ ]:
def compute_batch_metrics(pred, target, thr=0.5):
    pred = (torch.sigmoid(pred) > thr).float()
    intersection = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) - intersection
    
    dice = (2 * intersection + 1e-7) / (pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) + 1e-7)
    iou = (intersection + 1e-7) / (union + 1e-7)
    return dice.mean().item(), iou.mean().item()

# ----- TRAINING LOOP -----
best_dice = 0.0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss, val_dice, val_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            loss = combined_loss(outputs, masks)
            val_loss += loss.item() * images.size(0)
            
            d_m, i_m = compute_batch_metrics(outputs, masks)
            val_dice += d_m * images.size(0)
            val_iou += i_m * images.size(0)
            
    val_loss /= len(val_loader.dataset)
    val_dice /= len(val_loader.dataset)
    val_iou /= len(val_loader.dataset)
    
    scheduler.step(val_dice)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")
    
    if val_dice > best_dice:
        best_dice = val_dice
        # Save path can be configured to drive or local folder
        torch.save(model.state_dict(), "best_unet_resnet34_arcade.pth")
        print(f"==> Saved best model with Dice: {best_dice:.4f}")